SQL 版本：用 SQL 重做一遍核心分析

In [2]:
import sys, sqlite3
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

连接数据库

In [3]:
DB = ROOT / "data" / "processed" / "superstore.db"

con = sqlite3.connect(DB)

def run_sql(name):
    """读取 sql/<name>.sql 并执行，返回 DataFrame"""
    sql = (ROOT / "sql" / f"{name}.sql").read_text(encoding="utf-8")
    return pd.read_sql_query(sql, con)

每个折扣总销售额

In [4]:
run_sql("01_discount_by_band")

,Discount,Sales,dis_count
0,0.00,1.087908e+06,4798
1,0.10,5.436935e+04,94
2,0.15,2.755852e+04,52
3,0.20,7.645944e+05,3657
4,0.30,1.032267e+05,227
5,0.32,1.449346e+04,27
6,0.40,1.164178e+05,206
7,0.45,5.484974e+03,11
8,0.50,5.891854e+04,66
9,0.60,6.644700e+03,138


7折以上/以下 所占总销售额、利润的比例

In [5]:
run_sql("02_deep_discount_share")

,up_7_sales,down_7_sales,up_7_profit,down_7_profit
0,0.842082,0.157918,1.472687,-0.472687


亏损的类别

In [6]:
run_sql("03_loss_by_subcategory")

,Sub-Category,down_7
0,Binders,-38510.4964
1,Tables,-30698.2228
2,Machines,-29555.3521
3,Bookcases,-11097.7614
4,Appliances,-8629.6412
5,Chairs,-6737.1167
6,Phones,-6385.7854
7,Furnishings,-5944.6552


对比 pandas+ sql

In [8]:
from src.clean import load_clean

df = load_clean()
pd_share = df[df["Discount"] >= 0.3]["Profit"].sum() / df["Profit"].sum()

sql_share = run_sql("02_deep_discount_share").iloc[0]["down_7_profit"]

print(f"SQL {sql_share:.4f}")
print(f"pandas {pd_share:.4f}")

SQL -0.4727
pandas -0.4727


In [9]:
con.close()